# Crowd Video GRU Inference Notebook

This notebook runs the same crowd feature extraction and GRU inference pipeline used in `crowd_analytics`.

Run the cells in order:
1. Set your local video path.
2. Load the trained model, scaler, and label encoder.
3. Extract 11 crowd features per second.
4. Predict `SAFE` / `WARNING` / `HIGH`.
5. Export an annotated video and a predictions table.

In [1]:
from __future__ import annotations

import os
import sys
import math
import pickle
from collections import deque
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display, Video

# Make the repo importable from this notebook no matter where it is opened.
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "crowd_analytics").exists():
            return candidate
    raise FileNotFoundError("Could not find the crowd_analytics folder above the current notebook directory.")

PROJECT_ROOT = find_project_root()
CROWD_DIR = PROJECT_ROOT / "crowd_analytics"
if str(CROWD_DIR) not in sys.path:
    sys.path.insert(0, str(CROWD_DIR))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")

PROJECT_ROOT: C:\CCMPS\Crowd_Density\CCMPS\crowd_analytics\inference
Python: 3.13.3
TensorFlow: 2.21.0


## Load Configuration and Paths

The notebook uses the same 11 feature columns and 3 class labels as the trained GRU pipeline. It will automatically pick the first model/scaler/encoder file it finds in the repo.

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [3]:
import sys
from pathlib import Path

# notebook.ipynb -> inference
NOTEBOOK_DIR = Path.cwd()

# inference -> crowd_analytics
PROJECT_ROOT = NOTEBOOK_DIR.parent

# add crowd_analytics to python path
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\CCMPS\Crowd_Density\CCMPS\crowd_analytics


In [4]:
from utils.config import SEQUENCE_LENGTH, FEATURE_COLUMNS, RISK_CLASSES, VIDEO_FPS, PREDICTION_SMOOTHING_WINDOW
from analytics.trajectory_analytics import ZoneAnalytics
from detector.yolo_detector import YOLODetector
from tracker.deep_sort_tracker import DeepSortTracker

# Choose your local video here. Change this path to the file you want to process.
VIDEO_PATH = PROJECT_ROOT / "uploads" / "Crowd.mp4"
OUTPUT_DIR = PROJECT_ROOT / "crowd_analytics" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO_PATH = OUTPUT_DIR / "annotated_gru_output.mp4"
PREDICTIONS_CSV_PATH = OUTPUT_DIR / "gru_predictions.csv"

MODEL_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "models" / "gru_best.keras",
    PROJECT_ROOT / "models" / "gru_crowd_risk.keras",
]
SCALER_CANDIDATES = [
    PROJECT_ROOT/ "outputs" / "models" / "scaler.pkl",
    PROJECT_ROOT/ "models" / "feature_scaler.pkl",
    PROJECT_ROOT/ "models" / "scaler.pkl",
]
ENCODER_CANDIDATES = [
    PROJECT_ROOT  / "outputs" / "models" / "label_encoder.pkl",
    PROJECT_ROOT / "models" / "label_encoder.pkl",
]

def first_existing(paths: list[Path]) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(f"None of these files exist: {paths}")

MODEL_PATH = first_existing(MODEL_CANDIDATES)
SCALER_PATH = first_existing(SCALER_CANDIDATES)
ENCODER_PATH = first_existing(ENCODER_CANDIDATES)

print(f"MODEL_PATH:   {MODEL_PATH}")
print(f"SCALER_PATH:  {SCALER_PATH}")
print(f"ENCODER_PATH: {ENCODER_PATH}")
print(f"VIDEO_PATH:   {VIDEO_PATH}")
print(f"FEATURES:     {len(FEATURE_COLUMNS)}")
print(f"SEQUENCE_LEN: {SEQUENCE_LENGTH}")

MODEL_PATH:   c:\CCMPS\Crowd_Density\CCMPS\crowd_analytics\outputs\models\gru_best.keras
SCALER_PATH:  c:\CCMPS\Crowd_Density\CCMPS\crowd_analytics\outputs\models\scaler.pkl
ENCODER_PATH: c:\CCMPS\Crowd_Density\CCMPS\crowd_analytics\outputs\models\label_encoder.pkl
VIDEO_PATH:   c:\CCMPS\Crowd_Density\CCMPS\crowd_analytics\uploads\Crowd.mp4
FEATURES:     11
SEQUENCE_LEN: 6


## Initialize Video Input and Feature Extractor

This uses the same YOLOv8 + DeepSORT + ZoneAnalytics pipeline as the repo inference code.

In [5]:
VIDEO_PATH="c:\\CCMPS\\Crowd_Density\\CCMPS\\uploads\\Crowd.mp4"


cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise IOError(f"Could not open video: {VIDEO_PATH}")

source_fps = cap.get(cv2.CAP_PROP_FPS) or VIDEO_FPS
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1280)
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)

print(f"Source FPS: {source_fps:.2f}")
print(f"Frame size:  {frame_width} x {frame_height}")
print(f"Using model window: {SEQUENCE_LENGTH} seconds")

# Repo-native feature extractor and tracker.
detector = YOLODetector(model_name="crowd_analytics\\yolov8n.pt", conf_threshold=0.40, iou_threshold=0.45, device="cpu")
tracker = DeepSortTracker()
analytics = ZoneAnalytics()

cap.release()
print("YOLO detector, DeepSORT tracker, and ZoneAnalytics initialised.")

Source FPS: 30.00
Frame size:  1280 x 720
Using model window: 6 seconds
[YOLODetector] Loaded crowd_analytics\yolov8n.pt | conf=0.4 iou=0.45


ModuleNotFoundError: No module named 'pkg_resources'

## Load Trained Model and Preprocessors

The notebook tries the repo's `RealTimePredictor` first. If your local TensorFlow / Keras version cannot deserialize the saved `.keras` file, it falls back to a compatibility loader so the prediction pipeline still runs.

In [ ]:
from inference.realtime_predictor import RealTimePredictor

@tf.keras.utils.register_keras_serializable(package="CCMPS")
class Keras2BatchNormalization(tf.keras.layers.BatchNormalization):
    def __init__(self, **kwargs):
        kwargs.pop("renorm", None)
        kwargs.pop("renorm_clipping", None)
        kwargs.pop("renorm_momentum", None)
        super().__init__(**kwargs)


def load_compatible_model(model_path: Path):
    loaders = []

    # 1) Native loader for matching TF/Keras versions.
    loaders.append(("native", lambda: tf.keras.models.load_model(str(model_path), compile=False)))

    # 2) Legacy Keras package if installed.
    try:
        import tf_keras as legacy_keras
        loaders.append(("tf_keras", lambda: legacy_keras.models.load_model(str(model_path), compile=False)))
    except Exception:
        pass

    # 3) Custom BatchNormalization compatibility shim.
    loaders.append(("custom_objects", lambda: tf.keras.models.load_model(
        str(model_path),
        custom_objects={"BatchNormalization": Keras2BatchNormalization},
        compile=False,
    )))

    last_error = None
    for loader_name, loader in loaders:
        try:
            model = loader()
            print(f"Loaded GRU model with {loader_name} loader: {model_path}")
            return model
        except Exception as exc:
            last_error = exc
            print(f"{loader_name} loader failed: {exc}")

    raise RuntimeError(f"Unable to load model {model_path}") from last_error


class NotebookPredictor:
    def __init__(self, model_path: Path, scaler_path: Path, encoder_path: Path):
        self.model = load_compatible_model(model_path)
        with open(scaler_path, "rb") as f:
            self.scaler = pickle.load(f)
        with open(encoder_path, "rb") as f:
            self.encoder = pickle.load(f)
        self.buffer = deque(maxlen=SEQUENCE_LENGTH)
        self.history = deque(maxlen=PREDICTION_SMOOTHING_WINDOW)
        self.last_label = "SAFE"

    def predict(self, feature_dict: dict[str, float]):
        vec = np.array([feature_dict.get(c, 0.0) for c in FEATURE_COLUMNS], dtype=np.float32)
        vec_scaled = self.scaler.transform(vec.reshape(1, -1))[0]
        self.buffer.append(vec_scaled)

        if len(self.buffer) < SEQUENCE_LENGTH:
            probs = np.zeros(len(RISK_CLASSES), dtype=np.float32)
            probs[RISK_CLASSES.index(self.last_label)] = 1.0
            return self.last_label, probs, 1.0

        seq = np.array(self.buffer)[np.newaxis, ...]
        probs = self.model.predict(seq, verbose=0)[0]
        raw_label = RISK_CLASSES[int(np.argmax(probs))]
        self.history.append(raw_label)
        smoothed = max(set(self.history), key=list(self.history).count)
        self.last_label = smoothed
        return smoothed, probs, float(np.max(probs))

    def is_ready(self) -> bool:
        return len(self.buffer) >= SEQUENCE_LENGTH


try:
    predictor = RealTimePredictor(model_path=MODEL_PATH, scaler_path=SCALER_PATH, encoder_path=ENCODER_PATH)
    print("Using repo RealTimePredictor.")
except Exception as exc:
    print(f"RealTimePredictor failed to load in this environment: {exc}")
    print("Falling back to notebook-local compatible predictor.")
    predictor = NotebookPredictor(MODEL_PATH, SCALER_PATH, ENCODER_PATH)

print(f"Predictor ready: {predictor.__class__.__name__}")
print(f"Loaded encoder classes: {list(getattr(predictor.encoder, 'classes_', RISK_CLASSES))}")

## Process Video Frame-by-Frame

This cell runs YOLOv8 + DeepSORT on the uploaded video, aggregates the same 11 features used for training, and feeds them to the GRU once per second.

In [ ]:
RISK_COLORS = {
    "SAFE": (46, 204, 113),
    "WARNING": (243, 156, 18),
    "HIGH": (231, 76, 60),
}


def put_panel(frame, title, lines, color):
    overlay = frame.copy()
    x1, y1, w, h = 18, 16, 460, 130
    x2, y2 = x1 + w, y1 + h
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (18, 22, 30), -1)
    cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
    cv2.addWeighted(overlay, 0.82, frame, 0.18, 0, frame)
    cv2.putText(frame, title, (x1 + 14, y1 + 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    for idx, line in enumerate(lines):
        cv2.putText(frame, line, (x1 + 14, y1 + 62 + idx * 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (235, 235, 235), 1, cv2.LINE_AA)


def run_video_inference():
    cap_local = cv2.VideoCapture(str(VIDEO_PATH))
    if not cap_local.isOpened():
        raise IOError(f"Could not open video: {VIDEO_PATH}")

    fps_local = cap_local.get(cv2.CAP_PROP_FPS) or VIDEO_FPS
    frame_step = max(int(round(fps_local)), 1)
    width = int(cap_local.get(cv2.CAP_PROP_FRAME_WIDTH) or frame_width)
    height = int(cap_local.get(cv2.CAP_PROP_FRAME_HEIGHT) or frame_height)

    writer = cv2.VideoWriter(
        str(OUTPUT_VIDEO_PATH),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps_local,
        (width, height),
    )

    local_detector = detector
    local_tracker = DeepSortTracker()
    local_analytics = ZoneAnalytics()

    rows = []
    frame_idx = 0
    second_idx = 0
    current_label = "SAFE"
    current_probs = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    current_conf = 1.0

    print(f"Running inference on {VIDEO_PATH.name} at {fps_local:.2f} FPS")
    print(f"Features per second: {len(FEATURE_COLUMNS)}")

    while True:
        ret, frame = cap_local.read()
        if not ret:
            break

        detections = local_detector.detect(frame)
        tracks = local_tracker.update(detections, frame, frame_idx)
        track_states = local_tracker.get_all_states()
        local_analytics.ingest_frame(tracks, track_states, frame_idx)

        if frame_idx > 0 and frame_idx % frame_step == 0:
            features = local_analytics.flush_window()
            current_label, current_probs, current_conf = predictor.predict(features)
            second_idx += 1
            rows.append({
                "second": second_idx,
                **features,
                "risk_label": current_label,
                "confidence": float(current_conf),
                "prob_safe": float(current_probs[0]),
                "prob_warning": float(current_probs[1]),
                "prob_high": float(current_probs[2]),
            })
            print(
                f"[t={second_idx:4d}s] people={features['people_count']:.1f} "
                f"density={features['density']:.3f} speed={features['avg_speed_mps']:.2f} "
                f"→ {current_label} ({current_conf:.0%})"
            )

        for tid, x1, y1, x2, y2, cx, cy in tracks:
            risk_color = RISK_COLORS[current_label]
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), risk_color, 2)
            cv2.circle(frame, (int(cx), int(cy)), 2, risk_color, -1)

        label_text = f"{current_label}  {current_conf:.0%}"
        put_panel(
            frame,
            "GRU Crowd Risk",
            [
                f"Video: {VIDEO_PATH.name}",
                f"Second: {second_idx}",
                f"Prediction: {label_text}",
                f"Model: {MODEL_PATH.name}",
            ],
            RISK_COLORS[current_label],
        )

        cv2.putText(
            frame,
            f"FPS: {fps_local:.1f}",
            (width - 130, height - 18),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (220, 220, 220),
            1,
            cv2.LINE_AA,
        )

        writer.write(frame)
        frame_idx += 1

    cap_local.release()
    writer.release()

    results_df = pd.DataFrame(rows)
    results_df.to_csv(PREDICTIONS_CSV_PATH, index=False)
    print(f"Saved annotated video → {OUTPUT_VIDEO_PATH}")
    print(f"Saved predictions CSV → {PREDICTIONS_CSV_PATH}")
    return results_df


results_df = run_video_inference()
results_df.head()

## Visualize Results and Export Annotations

Review the prediction timeline, open the annotated video, and inspect the exported CSV.

In [ ]:
if results_df.empty:
    print("No prediction rows were generated. Check the video path, FPS, and detection quality.")
else:
    display(results_df[["second", "risk_label", "confidence", "people_count", "density", "avg_speed_mps"]].head(20))

    fig, ax = plt.subplots(figsize=(14, 5))
    plot_df = results_df.copy()
    for label in RISK_CLASSES:
        subset = plot_df[plot_df["risk_label"] == label]
        if subset.empty:
            continue
        rgb = tuple(np.array(RISK_COLORS[label])[::-1] / 255.0)
        ax.scatter(subset["second"], subset["confidence"], s=32, color=rgb, label=label, alpha=0.9)

    ax.plot(plot_df["second"], plot_df["confidence"], color="gray", alpha=0.25, linewidth=1)
    ax.set_title("GRU crowd risk over time")
    ax.set_xlabel("Second")
    ax.set_ylabel("Prediction confidence")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.2)
    ax.legend(title="Risk level")
    plt.show()

    print(f"Annotated video: {OUTPUT_VIDEO_PATH}")
    print(f"Predictions CSV:  {PREDICTIONS_CSV_PATH}")
    display(Video(str(OUTPUT_VIDEO_PATH), embed=True))